# Agentic BI with SQLite

This tutorial uses a fixed investigation to explain a revenue change in a fictional sales warehouse. It separates query results, interpretation, and optional fictional context. No LLM or online service is needed.

**מטרת התרגיל** — להתחיל במדד הכנסות נטו, לבחון איפה חל השינוי, ורק אז לבדוק אם הקשר חיצוני מסייע לגבש השערה.

## 1. Build the warehouse and verify the metric

The star schema contains dates, cities, product categories, and daily sales. Start with the stated metric and choose the region and periods.

In [ ]:
from pprint import pprint

from agentic_bi import DELIVERY_CONTEXT, NOISY_CONTEXT, NET_REVENUE, investigate
from sample_warehouse import build_warehouse

db = build_warehouse()
db.execute("PRAGMA query_only = ON")
region, baseline, current = "North", "2026-05", "2026-06"
print("Net revenue:", NET_REVENUE)
print("Synthetic fact rows:", db.execute("SELECT COUNT(*) FROM fact_daily_sales").fetchone()[0])

## 2. Compare the periods using warehouse evidence

The regional total and city changes come from SQLite. If revenue declines, follow the city with the largest negative change and inspect operational measures.

In [ ]:
warehouse_only = investigate(db, region, baseline, current)
pprint(warehouse_only, sort_dicts=False)

## 3. Test relevant context and irrelevant sources

Compare the same SQL evidence with a fictional delivery report. Then add reports for another topic, city, and period. An accepted source can support further investigation. It cannot establish a cause.

In [ ]:
with_context = investigate(db, region, baseline, current, (DELIVERY_CONTEXT,))
pprint(with_context, sort_dicts=False)

In [ ]:
with_noise = investigate(db, region, baseline, current, (DELIVERY_CONTEXT, *NOISY_CONTEXT))
print("Finding:", with_noise["finding"])
print("Accepted context:", with_noise["accepted_context"])
print("Sources excluded:", with_noise["excluded_context"])

## 4. Change the data and rerun

Use a separate in-memory copy to test how a higher June revenue changes the conclusion. The original sample remains available for comparison.

In [ ]:
from contextlib import closing

with closing(build_warehouse()) as alternative:
    alternative.execute('''
        UPDATE fact_daily_sales SET gross_revenue = gross_revenue * 3
        WHERE city_id = 1 AND date_id IN (
            SELECT date_id FROM dim_date WHERE month = '2026-06'
        )
    ''')
    alternative.execute("PRAGMA query_only = ON")
    counterfactual = investigate(alternative, region, baseline, current, (DELIVERY_CONTEXT,))
    print(counterfactual["finding"])
    print("Context accepted:", counterfactual["accepted_context"])
db.close()